# 03 — Full Training

Production training run on the full Fakeddit HDF5 (~12K train / 2.5K val).

Calls `training/train.py` which does two-stage fine-tuning (Blueprint §8.1):
- **Stage 1** (1 epoch, lr=1e-4): encoders frozen, head warmup (projections + fusion + classifier only).
- **Stage 2** (≤3 epochs, lr=2e-5): last 2 layers/blocks of each encoder unfrozen, early-stop on val F1 (patience=2).

Per-epoch checkpoints + a `best.pt` go to Drive (`cfg.checkpointing.dir`). TensorBoard event files go to `cfg.logging.tensorboard_dir/{run_name}/`. The notebook below launches the run, then opens TensorBoard inline so you can watch loss curves live.

## Bootstrap (idempotent — same cell as notebook 02)
Sets env vars, mounts Drive (pick **Account B** in OAuth), clones/pulls the repo, installs deps, removes `jax`/`flax` (they force `numpy>=2` and break the project pin), and copies the HDF5 to local SSD for faster random reads.

In [ ]:
# Bootstrap — idempotent. Safe to re-run on a fresh or warm Colab runtime.
import os, sys, subprocess, shutil

# Env vars FIRST — must be set before any transformers import.
os.environ['USE_FLAX'] = 'FALSE'
os.environ['USE_TF'] = 'FALSE'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
REPO_URL = 'https://github.com/staharizvi/hemt-clip-fnd.git'
REPO_DIR = '/content/hemt-clip-fnd'
H5_DRIVE = '/content/drive/MyDrive/hemt-clip-fnd/data/fakeddit.h5'
H5_LOCAL = '/content/fakeddit.h5'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('Drive already mounted.')

    if os.path.exists(os.path.join(REPO_DIR, '.git')):
        print('Repo present — pulling latest…')
        subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    else:
        print('Cloning repo…')
        subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)

    subprocess.run(['pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements.txt'], check=True)
    # jaxlib/jax/flax force numpy>=2 and clash with the project numpy pin.
    subprocess.run(['pip', 'uninstall', '-y', '-q', 'jax', 'jaxlib', 'flax'], check=False)

    if not os.path.exists(H5_LOCAL):
        if os.path.exists(H5_DRIVE):
            print(f'Copying {H5_DRIVE} -> {H5_LOCAL}…')
            shutil.copy(H5_DRIVE, H5_LOCAL)
        else:
            print(f'WARNING: {H5_DRIVE} not found — run the data-prep notebook first.')
    else:
        print(f'h5 already at {H5_LOCAL}.')

    os.chdir(REPO_DIR)

print('\ncwd:', os.getcwd())
print('h5 :', H5_LOCAL, 'exists:', os.path.exists(H5_LOCAL))

## Point the trainer at the local HDF5
The trainer reads `cfg.data.hdf5_path` from `configs/base.yaml`, which defaults to the Drive copy. For a real run we want the local copy — easier to override on the fly than to edit the config file.

In [ ]:
import yaml, json, pathlib
cfg_path = pathlib.Path('configs/base.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
if cfg['data']['hdf5_path'] != '/content/fakeddit.h5':
    cfg['data']['hdf5_path'] = '/content/fakeddit.h5'
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Patched cfg.data.hdf5_path -> /content/fakeddit.h5')
else:
    print('cfg already points at local HDF5.')
print('checkpoints :', cfg['checkpointing']['dir'])
print('tensorboard :', cfg['logging']['tensorboard_dir'])

## Launch TensorBoard (open before training so you can watch live)
On a fresh runtime the directory may not exist yet — that's fine, TB picks up new event files automatically.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/hemt-clip-fnd/runs

## Train
Run the full `hemt_clip` variant. Stage 1 = 1 epoch (head warmup), Stage 2 = up to 3 epochs (early-stop on val F1). Per-epoch checkpoints go to Drive — if the runtime dies, re-run with `--resume <last-ckpt>` (see the resume cell below).

In [ ]:
!python -m training.train --variant hemt_clip

## Resume (if the runtime disconnects)
Find the most recent checkpoint for this run and resume from it. Optimiser/scheduler/scaler/RNG state are all restored — the resumed run is bit-equivalent to one that never stopped.

In [ ]:
import glob, os
ckpts = sorted(glob.glob('/content/drive/MyDrive/hemt-clip-fnd/checkpoints/hemt_*_stage*_epoch*.pt'),
               key=os.path.getmtime)
if not ckpts:
    print('No checkpoint found — start a fresh run instead.')
else:
    print('latest ckpt:', ckpts[-1])
    # Uncomment to resume:
    # !python -m training.train --variant hemt_clip --resume "{ckpts[-1]}"

## After training
- The best model is at `{ckpt_dir}/{run_name}_best.pt` (saved each time val F1 improves).
- TensorBoard tab above shows `train/loss`, `train/lr`, `train_epoch/{loss,acc}`, `val/{loss,acc,f1,prec,rec}`, plus per-epoch weight/grad histograms.
- Next: `notebooks/04_evaluation.ipynb` for the held-out test metrics + confusion matrix + ROC, and then the ablation runner to repeat for `text_only`, `image_only`, `concat_fusion`.